In [2]:
import sys
print(sys.version)

import pandas as pd
import numpy as np

print("Data Wrangling environment ready!")

3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Data Wrangling environment ready!


In [3]:

from google.colab import files

uploaded = files.upload()


Saving ApexPlanet_DataAnalytics_Dataset.xlsx to ApexPlanet_DataAnalytics_Dataset.xlsx


In [4]:

print("Uploaded files:", list(uploaded.keys()))


Uploaded files: ['ApexPlanet_DataAnalytics_Dataset.xlsx']


In [6]:
file_name = list(uploaded.keys())[0]

# Since the file is an Excel spreadsheet (.xlsx), we must use read_excel instead of read_csv
df = pd.read_excel(file_name)

print("Dataset Shape:", df.shape)
print("\nFirst 5 Rows:")
display(df.head())

print("\nColumn Names:")
print(df.columns.tolist())

print("\nMissing Values:")
print(df.isnull().sum())

Dataset Shape: (1000, 12)

First 5 Rows:


,Order_ID,Order_Date,Customer_ID,Customer_Name,Age,Gender,City,Product,Category,Quantity,Unit_Price,Total_Sales
0,ORD100002,2025-02-25,CUST5529,Customer_227,30.0,Female,Bengaluru,Rice,Grocery,7,2829.77,19808.39
1,ORD100003,2025-10-14,CUST3127,Customer_182,63.0,Male,Bengaluru,Book,Education,5,27906.16,139530.80
2,ORD100004,2025-05-13,CUST8887,Customer_487,62.0,Female,Bengaluru,Book,Education,8,37491.06,299928.48
3,ORD100005,2025-12-02,CUST2515,Customer_470,65.0,Female,Kolkata,Mobile,Electronics,9,28541.36,256872.24
4,ORD100006,2025-11-20,CUST4796,Customer_380,44.0,Male,Bengaluru,Rice,Grocery,10,14036.59,140365.90



Column Names:
['Order_ID', 'Order_Date', 'Customer_ID', 'Customer_Name', 'Age', 'Gender', 'City', 'Product', 'Category', 'Quantity', 'Unit_Price', 'Total_Sales']

Missing Values:
Order_ID          0
Order_Date        0
Customer_ID       0
Customer_Name     0
Age              20
Gender            0
City             13
Product           0
Category          0
Quantity          0
Unit_Price        0
Total_Sales       0
dtype: int64


In [8]:

# Read every worksheet without forcing column types
sheets = pd.read_excel(
    file_name,
    sheet_name=None,
    dtype=object,
    keep_default_na=False
)

print("Worksheets found:", list(sheets.keys()))

for sheet_name, data in sheets.items():
    print(f"\n--- Sheet: {sheet_name} ---")
    print("Rows and columns:", data.shape)
    print("Column names:", data.columns.tolist())
    print("\nData types:")
    print(data.dtypes)
    print("\nFirst 5 rows:")
    display(data.head())


Worksheets found: ['Sales_Dataset']

--- Sheet: Sales_Dataset ---
Rows and columns: (1000, 12)
Column names: ['Order_ID', 'Order_Date', 'Customer_ID', 'Customer_Name', 'Age', 'Gender', 'City', 'Product', 'Category', 'Quantity', 'Unit_Price', 'Total_Sales']

Data types:
Order_ID         object
Order_Date       object
Customer_ID      object
Customer_Name    object
Age              object
Gender           object
City             object
Product          object
Category         object
Quantity         object
Unit_Price       object
Total_Sales      object
dtype: object

First 5 rows:


,Order_ID,Order_Date,Customer_ID,Customer_Name,Age,Gender,City,Product,Category,Quantity,Unit_Price,Total_Sales
0,ORD100002,2025-02-25,CUST5529,Customer_227,30,Female,Bengaluru,Rice,Grocery,7,2829.77,19808.39
1,ORD100003,2025-10-14,CUST3127,Customer_182,63,Male,Bengaluru,Book,Education,5,27906.16,139530.8
2,ORD100004,2025-05-13,CUST8887,Customer_487,62,Female,Bengaluru,Book,Education,8,37491.06,299928.48
3,ORD100005,2025-12-02,CUST2515,Customer_470,65,Female,Kolkata,Mobile,Electronics,9,28541.36,256872.24
4,ORD100006,2025-11-20,CUST4796,Customer_380,44,Male,Bengaluru,Rice,Grocery,10,14036.59,140365.9


In [9]:

# Create a working copy
clean_df = df.copy()

# Remove extra spaces from column names
clean_df.columns = clean_df.columns.str.strip()

# Remove extra spaces from text columns
for col in clean_df.select_dtypes(include=["object"]).columns:
    clean_df[col] = clean_df[col].apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )

# Check missing values
print("Missing values before cleaning:")
print(clean_df.isnull().sum())

# Check duplicate rows
print("\nDuplicate rows:", clean_df.duplicated().sum())

# Remove exact duplicate rows
clean_df = clean_df.drop_duplicates()

print("\nRows after removing duplicates:", len(clean_df))


Missing values before cleaning:
Order_ID          0
Order_Date        0
Customer_ID       0
Customer_Name     0
Age              20
Gender            0
City             13
Product           0
Category          0
Quantity          0
Unit_Price        0
Total_Sales       0
dtype: int64

Duplicate rows: 0

Rows after removing duplicates: 1000


In [10]:

# Convert date only after checking its current representation
clean_df["Order_Date"] = pd.to_datetime(
    clean_df["Order_Date"], errors="coerce"
)

# Convert numeric columns safely
numeric_cols = [
    "Age", "Quantity", "Unit_Price", "Total_Sales"
]

for col in numeric_cols:
    clean_df[col] = pd.to_numeric(
        clean_df[col], errors="coerce"
    )

# Validate sales calculations
expected_sales = clean_df["Quantity"] * clean_df["Unit_Price"]

mismatches = (
    clean_df["Total_Sales"] - expected_sales
).abs().gt(0.01)

print("Invalid dates:", clean_df["Order_Date"].isna().sum())
print("Invalid numeric values:")
print(clean_df[numeric_cols].isna().sum())
print("Sales calculation mismatches:", mismatches.sum())


Invalid dates: 0
Invalid numeric values:
Age            20
Quantity        0
Unit_Price      0
Total_Sales     0
dtype: int64
Sales calculation mismatches: 0


In [11]:

age_numeric = pd.to_numeric(clean_df["Age"], errors="coerce")

invalid_age = clean_df.loc[
    age_numeric.isna() & clean_df["Age"].notna(),
    ["Order_ID", "Customer_Name", "Age"]
]

print("Invalid Age entries:", len(invalid_age))
display(invalid_age)


Invalid Age entries: 0


,Order_ID,Customer_Name,Age


In [12]:
age_numeric = pd.to_numeric(clean_df["Age"], errors="coerce")

invalid_range = age_numeric.notna() & (
    (age_numeric < 0) | (age_numeric > 100)
)

print("Age values outside 0–100:", invalid_range.sum())
display(clean_df.loc[invalid_range, ["Order_ID", "Age"]])

Age values outside 0–100: 0


,Order_ID,Age


In [13]:

print("Age data type:", clean_df["Age"].dtype)
print("\nMissing Age values:", clean_df["Age"].isna().sum())
print("\nAge summary:")
print(clean_df["Age"].describe())

print("\nAge values outside 0–100:")
age = pd.to_numeric(clean_df["Age"], errors="coerce")
print(((age < 0) | (age > 100)).sum())


Age data type: float64

Missing Age values: 20

Age summary:
count    980.000000
mean      41.360204
std       13.822597
min       18.000000
25%       30.000000
50%       41.000000
75%       54.000000
max       65.000000
Name: Age, dtype: float64

Age values outside 0–100:
0


In [14]:

# Check missing Age records
missing_age = clean_df[clean_df["Age"].isna()]

print("Records with missing Age:", len(missing_age))
display(missing_age[["Order_ID", "Customer_ID", "Customer_Name", "Age"]])


Records with missing Age: 20


,Order_ID,Customer_ID,Customer_Name,Age
48,ORD100050,CUST5523,Customer_432,NaN
98,ORD100100,CUST7462,Customer_379,NaN
148,ORD100150,CUST4132,Customer_47,NaN
198,ORD100200,CUST2953,Customer_435,NaN
248,ORD100250,CUST3169,Customer_138,NaN
298,ORD100300,CUST7944,Customer_282,NaN
348,ORD100350,CUST9709,Customer_409,NaN
398,ORD100400,CUST5102,Customer_416,NaN
448,ORD100450,CUST6714,Customer_31,NaN
498,ORD100500,CUST2835,Customer_468,NaN


In [15]:

# Preserve the number of missing ages for reporting
missing_age_count = clean_df["Age"].isna().sum()

# Fill missing Age values using the median
age_median = clean_df["Age"].median()

clean_df["Age"] = clean_df["Age"].fillna(age_median)

print("Missing Age values before:", missing_age_count)
print("Median Age used:", age_median)
print("Missing Age values after:", clean_df["Age"].isna().sum())


Missing Age values before: 20
Median Age used: 41.0
Missing Age values after: 0


In [16]:
print("Missing values in all columns:")
print(clean_df.isna().sum())

Missing values in all columns:
Order_ID          0
Order_Date        0
Customer_ID       0
Customer_Name     0
Age               0
Gender            0
City             13
Product           0
Category          0
Quantity          0
Unit_Price        0
Total_Sales       0
dtype: int64


In [17]:

print("Missing City values:", clean_df["City"].isna().sum())

missing_city = clean_df[clean_df["City"].isna()]

display(
    missing_city[
        ["Order_ID", "Customer_ID", "Customer_Name", "City"]
    ]
)


Missing City values: 13


,Order_ID,Customer_ID,Customer_Name,City
73,ORD100075,CUST2157,Customer_31,NaN
148,ORD100150,CUST4132,Customer_47,NaN
223,ORD100225,CUST1443,Customer_315,NaN
298,ORD100300,CUST7944,Customer_282,NaN
373,ORD100375,CUST5835,Customer_362,NaN
448,ORD100450,CUST6714,Customer_31,NaN
523,ORD100525,CUST1090,Customer_316,NaN
598,ORD100050,CUST8859,Customer_53,NaN
673,ORD100675,CUST8502,Customer_220,NaN
748,ORD100750,CUST3730,Customer_273,NaN


In [18]:

print(clean_df["City"].value_counts(dropna=False))


City
Patna        135
Kolkata      133
Mumbai       131
Delhi        125
Hyderabad    125
Bengaluru    122
Gaya         117
Pune          99
NaN           13
Name: count, dtype: int64


In [19]:

clean_df["City"] = clean_df["City"].replace(
    r"^\s*$", np.nan, regex=True
)

clean_df["City"] = clean_df["City"].fillna("Unknown")

print("Missing City values after cleaning:",
      clean_df["City"].isna().sum())

print("Unknown City records:",
      (clean_df["City"] == "Unknown").sum())


Missing City values after cleaning: 0
Unknown City records: 13


In [20]:

print("Duplicate rows:", clean_df.duplicated().sum())

print("Duplicate Order_IDs:",
      clean_df["Order_ID"].duplicated().sum())


Duplicate rows: 0
Duplicate Order_IDs: 8


In [21]:

expected_sales = clean_df["Quantity"] * clean_df["Unit_Price"]

mismatch = (
    clean_df["Total_Sales"] - expected_sales
).abs() > 0.01

print("Sales mismatches:", mismatch.sum())


Sales mismatches: 0


In [22]:

duplicate_orders = clean_df[
    clean_df["Order_ID"].duplicated(keep=False)
].sort_values("Order_ID")

print("Rows with repeated Order_IDs:",
      len(duplicate_orders))

display(duplicate_orders)


Rows with repeated Order_IDs: 9


,Order_ID,Order_Date,Customer_ID,Customer_Name,Age,Gender,City,Product,Category,Quantity,Unit_Price,Total_Sales
48,ORD100050,2025-10-03,CUST5523,Customer_432,41.0,Male,Pune,Chair,Furniture,9,30834.87,277513.83
118,ORD100050,2025-01-22,CUST3194,Customer_114,34.0,Male,Kolkata,Book,Education,2,25795.55,51591.10
238,ORD100050,2025-03-01,CUST8467,Customer_104,23.0,Male,Bengaluru,Book,Education,3,2860.83,8582.49
358,ORD100050,2025-02-11,CUST6467,Customer_181,24.0,Female,Pune,Rice,Grocery,7,45480.39,318362.73
478,ORD100050,2025-11-22,CUST8225,Customer_468,64.0,Male,Bengaluru,Rice,Grocery,8,38957.59,311660.72
598,ORD100050,2025-02-21,CUST8859,Customer_53,41.0,Male,Unknown,Laptop,Electronics,7,20144.84,141013.88
718,ORD100050,2025-11-06,CUST7824,Customer_190,31.0,Female,Mumbai,Rice,Grocery,4,42642.59,170570.36
838,ORD100050,2025-08-03,CUST5585,Customer_289,35.0,Male,Kolkata,Chair,Furniture,9,28747.47,258727.23
958,ORD100050,2025-02-09,CUST7188,Customer_89,62.0,Male,Bengaluru,Mobile,Electronics,1,34619.77,34619.77


In [23]:

order_counts = clean_df["Order_ID"].value_counts()

print("Number of repeated Order_ID values:",
      (order_counts > 1).sum())

print("\nFrequency of repeated IDs:")
print(order_counts[order_counts > 1])


Number of repeated Order_ID values: 1

Frequency of repeated IDs:
Order_ID
ORD100050    9
Name: count, dtype: int64


In [24]:

order_050 = clean_df[
    clean_df["Order_ID"] == "ORD100050"
]

display(order_050)

print("Number of rows:", len(order_050))
print("Unique products:", order_050["Product"].nunique())
print("Unique customers:", order_050["Customer_ID"].nunique())
print("Unique order dates:", order_050["Order_Date"].nunique())


,Order_ID,Order_Date,Customer_ID,Customer_Name,Age,Gender,City,Product,Category,Quantity,Unit_Price,Total_Sales
48,ORD100050,2025-10-03,CUST5523,Customer_432,41.0,Male,Pune,Chair,Furniture,9,30834.87,277513.83
118,ORD100050,2025-01-22,CUST3194,Customer_114,34.0,Male,Kolkata,Book,Education,2,25795.55,51591.10
238,ORD100050,2025-03-01,CUST8467,Customer_104,23.0,Male,Bengaluru,Book,Education,3,2860.83,8582.49
358,ORD100050,2025-02-11,CUST6467,Customer_181,24.0,Female,Pune,Rice,Grocery,7,45480.39,318362.73
478,ORD100050,2025-11-22,CUST8225,Customer_468,64.0,Male,Bengaluru,Rice,Grocery,8,38957.59,311660.72
598,ORD100050,2025-02-21,CUST8859,Customer_53,41.0,Male,Unknown,Laptop,Electronics,7,20144.84,141013.88
718,ORD100050,2025-11-06,CUST7824,Customer_190,31.0,Female,Mumbai,Rice,Grocery,4,42642.59,170570.36
838,ORD100050,2025-08-03,CUST5585,Customer_289,35.0,Male,Kolkata,Chair,Furniture,9,28747.47,258727.23
958,ORD100050,2025-02-09,CUST7188,Customer_89,62.0,Male,Bengaluru,Mobile,Electronics,1,34619.77,34619.77


Number of rows: 9
Unique products: 5
Unique customers: 9
Unique order dates: 9


In [25]:

cols = [
    "Order_ID", "Order_Date", "Customer_ID",
    "Customer_Name", "Product", "Quantity",
    "Unit_Price", "Total_Sales"
]

display(order_050[cols].sort_values("Order_Date"))


,Order_ID,Order_Date,Customer_ID,Customer_Name,Product,Quantity,Unit_Price,Total_Sales
118,ORD100050,2025-01-22,CUST3194,Customer_114,Book,2,25795.55,51591.10
958,ORD100050,2025-02-09,CUST7188,Customer_89,Mobile,1,34619.77,34619.77
358,ORD100050,2025-02-11,CUST6467,Customer_181,Rice,7,45480.39,318362.73
598,ORD100050,2025-02-21,CUST8859,Customer_53,Laptop,7,20144.84,141013.88
238,ORD100050,2025-03-01,CUST8467,Customer_104,Book,3,2860.83,8582.49
838,ORD100050,2025-08-03,CUST5585,Customer_289,Chair,9,28747.47,258727.23
48,ORD100050,2025-10-03,CUST5523,Customer_432,Chair,9,30834.87,277513.83
718,ORD100050,2025-11-06,CUST7824,Customer_190,Rice,4,42642.59,170570.36
478,ORD100050,2025-11-22,CUST8225,Customer_468,Rice,8,38957.59,311660.72


In [26]:

quality_issues = pd.DataFrame([
    {
        "Column": "Order_ID",
        "Issue": "Same Order_ID linked to multiple customers and dates",
        "Affected_Value": "ORD100050",
        "Affected_Rows": 9,
        "Action": "Flagged for source verification; no rows deleted"
    }
])

display(quality_issues)


,Column,Issue,Affected_Value,Affected_Rows,Action
0,Order_ID,Same Order_ID linked to multiple customers and...,ORD100050,9,Flagged for source verification; no rows deleted


In [27]:

# Flag repeated Order_IDs without deleting records
order_counts = clean_df["Order_ID"].value_counts()

repeated_ids = order_counts[order_counts > 1].index

clean_df["Duplicate_Order_ID_Flag"] = (
    clean_df["Order_ID"].isin(repeated_ids)
)

print("Rows flagged:",
      clean_df["Duplicate_Order_ID_Flag"].sum())

display(
    clean_df[clean_df["Duplicate_Order_ID_Flag"]]
)


Rows flagged: 9


,Order_ID,Order_Date,Customer_ID,Customer_Name,Age,Gender,City,Product,Category,Quantity,Unit_Price,Total_Sales,Duplicate_Order_ID_Flag
48,ORD100050,2025-10-03,CUST5523,Customer_432,41.0,Male,Pune,Chair,Furniture,9,30834.87,277513.83,True
118,ORD100050,2025-01-22,CUST3194,Customer_114,34.0,Male,Kolkata,Book,Education,2,25795.55,51591.10,True
238,ORD100050,2025-03-01,CUST8467,Customer_104,23.0,Male,Bengaluru,Book,Education,3,2860.83,8582.49,True
358,ORD100050,2025-02-11,CUST6467,Customer_181,24.0,Female,Pune,Rice,Grocery,7,45480.39,318362.73,True
478,ORD100050,2025-11-22,CUST8225,Customer_468,64.0,Male,Bengaluru,Rice,Grocery,8,38957.59,311660.72,True
598,ORD100050,2025-02-21,CUST8859,Customer_53,41.0,Male,Unknown,Laptop,Electronics,7,20144.84,141013.88,True
718,ORD100050,2025-11-06,CUST7824,Customer_190,31.0,Female,Mumbai,Rice,Grocery,4,42642.59,170570.36,True
838,ORD100050,2025-08-03,CUST5585,Customer_289,35.0,Male,Kolkata,Chair,Furniture,9,28747.47,258727.23,True
958,ORD100050,2025-02-09,CUST7188,Customer_89,62.0,Male,Bengaluru,Mobile,Electronics,1,34619.77,34619.77,True


In [28]:

expected_sales = clean_df["Quantity"] * clean_df["Unit_Price"]

sales_mismatch = (
    clean_df["Total_Sales"] - expected_sales
).abs() > 0.01

print("Sales mismatches:", sales_mismatch.sum())


Sales mismatches: 0


In [29]:

quality_report = pd.DataFrame({
    "Check": [
        "Total rows",
        "Total columns",
        "Exact duplicate rows",
        "Missing values remaining",
        "Rows with repeated Order_ID",
        "Sales calculation mismatches"
    ],
    "Result": [
        len(clean_df),
        len(clean_df.columns),
        int(clean_df.drop(
            columns=["Duplicate_Order_ID_Flag"],
            errors="ignore"
        ).duplicated().sum()),
        int(clean_df.isna().sum().sum()),
        int(clean_df["Duplicate_Order_ID_Flag"].sum()),
        int(sales_mismatch.sum())
    ]
})

display(quality_report)


,Check,Result
0,Total rows,1000
1,Total columns,13
2,Exact duplicate rows,0
3,Missing values remaining,0
4,Rows with repeated Order_ID,9
5,Sales calculation mismatches,0


In [30]:

clean_df.to_excel("cleaned_sales_dataset.xlsx", index=False)
clean_df.to_csv("cleaned_sales_dataset.csv", index=False)
quality_report.to_csv("data_quality_report.csv", index=False)

print("Files created successfully!")


Files created successfully!


In [31]:

from google.colab import files

files.download("cleaned_sales_dataset.xlsx")
files.download("cleaned_sales_dataset.csv")
files.download("data_quality_report.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [32]:

print("FINAL DATA QUALITY CHECK")
print("-" * 35)

print("Dataset shape:", clean_df.shape)
print("Missing values:", clean_df.isna().sum().sum())
print("Exact duplicate rows:", clean_df.duplicated().sum())
print(
    "Rows flagged for repeated Order_ID:",
    clean_df["Duplicate_Order_ID_Flag"].sum()
)
print("Sales mismatches:", sales_mismatch.sum())

print("\nData types:")
print(clean_df.dtypes)


FINAL DATA QUALITY CHECK
-----------------------------------
Dataset shape: (1000, 13)
Missing values: 0
Exact duplicate rows: 0
Rows flagged for repeated Order_ID: 9
Sales mismatches: 0

Data types:
Order_ID                           object
Order_Date                 datetime64[ns]
Customer_ID                        object
Customer_Name                      object
Age                               float64
Gender                             object
City                               object
Product                            object
Category                           object
Quantity                            int64
Unit_Price                        float64
Total_Sales                       float64
Duplicate_Order_ID_Flag              bool
dtype: object


In [33]:

# Export cleaned dataset and quality report

clean_df.to_csv(
    "cleaned_sales_dataset.csv",
    index=False
)

clean_df.to_excel(
    "cleaned_sales_dataset.xlsx",
    index=False
)

quality_report.to_csv(
    "data_quality_report.csv",
    index=False
)

print("All project files exported successfully!")


All project files exported successfully!


In [34]:
from google.colab import files

files.download("cleaned_sales_dataset.xlsx")
files.download("cleaned_sales_dataset.csv")
files.download("data_quality_report.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>